In [2]:
import tensorflow as tf
import h5py
import numpy as np
import json

train_idx = np.load("../data/processed/train_idx.npy")
val_idx = np.load("../data/processed/val_idx.npy")
test_idx = np.load("../data/processed/test_idx.npy")
encoded_labels = np.load("../data/processed/encoded_labels.npy")

with open("../data/processed/class_weights.json") as f:
    class_weight_dict = {int(k): v for k, v in json.load(f).items()}

h5_path = "../data/raw/TheCycloneImageDataset/Cyclone_Images.h5"
labels_path = "../data/raw/TheCycloneImageDataset/Cyclone_Labels h5.npy"

# Raw labels se wind_speed aur pressure nikaalo (regression targets)
raw_labels = np.load(labels_path, allow_pickle=True)
wind_speeds = raw_labels[:, 5].astype(float)
pressures = raw_labels[:, 7].astype(float)

print("Wind speed range:", wind_speeds.min(), "-", wind_speeds.max())
print("Pressure range:", pressures.min(), "-", pressures.max())

Wind speed range: 10.0 - 168.0
Pressure range: 879.0 - 1024.0


In [3]:
wind_min, wind_max = wind_speeds.min(), wind_speeds.max()
pressure_min, pressure_max = pressures.min(), pressures.max()

wind_speeds_norm = (wind_speeds - wind_min) / (wind_max - wind_min)
pressures_norm = (pressures - pressure_min) / (pressure_max - pressure_min)

# In normalization values ko save kar lo - baad me predictions ko wapas asli scale me convert karne ke liye chahiye honge
norm_params = {
    "wind_min": float(wind_min), "wind_max": float(wind_max),
    "pressure_min": float(pressure_min), "pressure_max": float(pressure_max)
}
with open("../data/processed/regression_norm_params.json", "w") as f:
    json.dump(norm_params, f, indent=2)

print("Normalized wind speed sample:", wind_speeds_norm[:5])

Normalized wind speed sample: [0.12658228 0.12658228 0.12658228 0.15822785 0.15822785]


In [4]:
class CycloneDataGeneratorRegression(tf.keras.utils.Sequence):
    def __init__(self, h5_path, indices, class_labels, wind_norm, pressure_norm, batch_size=32, shuffle=True):
        self.h5_path = h5_path
        self.indices = indices
        self.class_labels = class_labels
        self.wind_norm = wind_norm
        self.pressure_norm = pressure_norm
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return len(self.indices) // self.batch_size

    def __getitem__(self, idx):
        batch_indices = self.indices[idx*self.batch_size : (idx+1)*self.batch_size]
        batch_indices_sorted = np.sort(batch_indices)

        with h5py.File(self.h5_path, 'r') as f:
            images = f['Images'][batch_indices_sorted].astype('float32')

        images = images[:, :, :, :3]
        images = tf.image.resize(images, (224, 224))
        images = tf.keras.applications.resnet50.preprocess_input(images)

        batch_class_labels = self.class_labels[batch_indices_sorted]
        batch_wind = self.wind_norm[batch_indices_sorted]
        batch_pressure = self.pressure_norm[batch_indices_sorted]

        # Dono regression targets ek saath combine karo (shape: batch_size, 2)
        batch_regression = np.stack([batch_wind, batch_pressure], axis=1)

        return images, {"classification": batch_class_labels, "regression": batch_regression}

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

train_gen_reg = CycloneDataGeneratorRegression(h5_path, train_idx, encoded_labels, wind_speeds_norm, pressures_norm, batch_size=32)
val_gen_reg = CycloneDataGeneratorRegression(h5_path, val_idx, encoded_labels, wind_speeds_norm, pressures_norm, batch_size=32, shuffle=False)

print("Batches:", len(train_gen_reg))

Batches: 461


In [5]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models, Model

num_classes = len(np.unique(encoded_labels))

base_model_reg = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model_reg.trainable = False   # pehle frozen rakho, baad me fine-tune karenge

inputs = layers.Input(shape=(224, 224, 3))
x = base_model_reg(inputs)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)

# Head 1: Classification (6 categories)
classification_output = layers.Dense(num_classes, activation='softmax', name='classification')(x)

# Head 2: Regression (wind speed + pressure, dono ek saath, 0-1 range me)
regression_output = layers.Dense(2, activation='sigmoid', name='regression')(x)

model_dual = Model(inputs=inputs, outputs=[classification_output, regression_output])

model_dual.summary()

2026-09-02 00:47:38.046935: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-09-02 00:47:38.046967: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-09-02 00:47:38.046972: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-09-02 00:47:38.047140: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-02 00:47:38.047149: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ input_layer_1[0]… │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │    262,272 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classification      │ (None, 6)         │        774 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ regression (Dense)  │ (None, 2)         │        258 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,851,016 (90.98 MB)

 Trainable params: 263,304 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [6]:
from tensorflow.keras.optimizers import Adam

model_dual.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss={
        'classification': 'sparse_categorical_crossentropy',
        'regression': 'mse'   # Mean Squared Error - regression ke liye standard loss
    },
    loss_weights={
        'classification': 1.0,
        'regression': 1.0
    },
    metrics={
        'classification': 'accuracy',
        'regression': 'mae'   # Mean Absolute Error - kitna galat predict kiya (average)
    }
)

In [7]:
history_dual = model_dual.fit(
    train_gen_reg,
    validation_data=val_gen_reg,
    epochs=10
)

Epoch 1/10


/Users/chessinorbit/Desktop/CYCLONE-PREDICTION/venv/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
2026-09-02 00:47:39.953266: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


461/461 ━━━━━━━━━━━━━━━━━━━━ 127s 270ms/step - classification_accuracy: 0.3819 - classification_loss: 2.2812 - loss: 2.3732 - regression_loss: 0.0920 - regression_mae: 0.2346 - val_classification_accuracy: 0.5035 - val_classification_loss: 1.2350 - val_loss: 1.3102 - val_regression_loss: 0.0752 - val_regression_mae: 0.2180
Epoch 2/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 145s 314ms/step - classification_accuracy: 0.4287 - classification_loss: 2.1073 - loss: 2.1787 - regression_loss: 0.0714 - regression_mae: 0.2099 - val_classification_accuracy: 0.5214 - val_classification_loss: 1.1632 - val_loss: 1.2112 - val_regression_loss: 0.0480 - val_regression_mae: 0.1821
Epoch 3/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 153s 331ms/step - classification_accuracy: 0.4509 - classification_loss: 1.9429 - loss: 2.0084 - regression_loss: 0.0655 - regression_mae: 0.2030 - val_classification_accuracy: 0.5411 - val_classification_loss: 1.0922 - val_loss: 1.1289 - val_regression_loss: 0.0367 - val_regression_mae: 0.1601
Epo

In [8]:
model_dual.save("../models/resnet_dual_head.h5")
print("Dual-head model saved!")

Dual-head model saved!


In [9]:
print(history_dual.history.keys())
print("Final regression MAE (train):", history_dual.history['regression_mae'][-1])
print("Final regression MAE (val):", history_dual.history['val_regression_mae'][-1])

dict_keys(['classification_accuracy', 'classification_loss', 'loss', 'regression_loss', 'regression_mae', 'val_classification_accuracy', 'val_classification_loss', 'val_loss', 'val_regression_loss', 'val_regression_mae'])
Final regression MAE (train): 0.17601542174816132
Final regression MAE (val): 0.1497531682252884


In [10]:
base_model_reg.trainable = True
for layer in base_model_reg.layers[:-20]:
    layer.trainable = False

model_dual.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss={'classification': 'sparse_categorical_crossentropy', 'regression': 'mse'},
    metrics={'classification': 'accuracy', 'regression': 'mae'}
)

history_dual_finetune = model_dual.fit(
    train_gen_reg,
    validation_data=val_gen_reg,
    epochs=10
)

Epoch 1/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 207s 440ms/step - classification_accuracy: 0.5355 - classification_loss: 1.1583 - loss: 1.1964 - regression_loss: 0.0381 - regression_mae: 0.1520 - val_classification_accuracy: 0.6110 - val_classification_loss: 0.9190 - val_loss: 0.9436 - val_regression_loss: 0.0246 - val_regression_mae: 0.1252
Epoch 2/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 201s 437ms/step - classification_accuracy: 0.5811 - classification_loss: 1.0223 - loss: 1.0578 - regression_loss: 0.0355 - regression_mae: 0.1483 - val_classification_accuracy: 0.6352 - val_classification_loss: 0.8677 - val_loss: 0.8923 - val_regression_loss: 0.0245 - val_regression_mae: 0.1264
Epoch 3/10
461/461 ━━━━━━━━━━━━━━━━━━━━ 201s 437ms/step - classification_accuracy: 0.6283 - classification_loss: 0.8963 - loss: 0.9304 - regression_loss: 0.0341 - regression_mae: 0.1455 - val_classification_accuracy: 0.6754 - val_classification_loss: 0.7955 - val_loss: 0.8202 - val_regression_loss: 0.0247 - val_regression_mae:

In [11]:
model_dual.save("../models/resnet_dual_finetuned.h5")
print("Fine-tuned dual-head model saved!")

Fine-tuned dual-head model saved!


In [12]:
print(history_dual_finetune.history.keys())
print("Final regression MAE (train):", history_dual_finetune.history['regression_mae'][-1])
print("Final regression MAE (val):", history_dual_finetune.history['val_regression_mae'][-1])
print("Final classification accuracy (val):", history_dual_finetune.history['val_classification_accuracy'][-1])

dict_keys(['classification_accuracy', 'classification_loss', 'loss', 'regression_loss', 'regression_mae', 'val_classification_accuracy', 'val_classification_loss', 'val_loss', 'val_regression_loss', 'val_regression_mae'])
Final regression MAE (train): 0.14530588686466217
Final regression MAE (val): 0.12531447410583496
Final classification accuracy (val): 0.8788265585899353


In [13]:
test_gen_reg = CycloneDataGeneratorRegression(h5_path, test_idx, encoded_labels, wind_speeds_norm, pressures_norm, batch_size=32, shuffle=False)

results = model_dual.evaluate(test_gen_reg)
print("Test results:", results)
print(model_dual.metrics_names)

98/98 ━━━━━━━━━━━━━━━━━━━━ 29s 295ms/step - classification_accuracy: 0.8763 - classification_loss: 0.4102 - loss: 0.4328 - regression_loss: 0.0226 - regression_mae: 0.1253
Test results: [0.4328359365463257, 0.4102385342121124, 0.022597359493374825, 0.8762755393981934, 0.12528415024280548]
['loss', 'compile_metrics', 'classification_loss', 'regression_loss']


In [14]:
wind_range = norm_params["wind_max"] - norm_params["wind_min"]
pressure_range = norm_params["pressure_max"] - norm_params["pressure_min"]

print(f"Wind speed MAE (approx): {0.1337 * wind_range:.2f} knots")
print(f"Note: This is combined MAE across both wind & pressure — for per-target breakdown, see below")

Wind speed MAE (approx): 21.12 knots
Note: This is combined MAE across both wind & pressure — for per-target breakdown, see below


In [15]:
print(model_dual)

<Functional name=functional, built=True>


In [16]:
model_dual.save("../models/resnet_dual_finetuned.keras")
print("Model saved in new .keras format!")

Model saved in new .keras format!
